In [ ]:
# Проба канала R15: можно ли вообще опубликовать датасет ИЗНУТРИ ядра Kaggle
# (авторизация CLI/API + symlink +_BIG file path). Ровно тот код-путь, что стоит в
# kernel-exp27c/_publish_layer, только на 2 МБ-заглушке: терять 3.6 GPU-часа на
# неподтверждённом канале нельзя. CPU-ядро, GPU-слоты не занимает.
import json, os, shutil, subprocess, hashlib

owner, slug = "voroninmichael", "prism-ds-publish-probe"
stage = f"/tmp/ds-{slug}"
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage)

src = "/kaggle/working/probe_payload.bin"
with open(src, "wb") as f:
    f.write(hashlib.sha256(b"prism").digest() * 32768)   # 1 MiB детерминированного мусора
print("payload", os.path.getsize(src), "bytes, sha", hashlib.sha256(open(src, 'rb').read()).hexdigest()[:16])

dst = os.path.join(stage, os.path.basename(src))
try:
    os.symlink(src, dst)
    print("symlink ok ->", os.path.realpath(dst), "isfile", os.path.isfile(dst))
except OSError as e:
    shutil.move(src, dst)
    print("symlink недоступен, перенесли:", repr(e))

with open(os.path.join(stage, "dataset-metadata.json"), "w") as f:
    json.dump({"title": "Prism: dataset publish probe (R15 channel)", "id": f"{owner}/{slug}",
               "is_private": True, "resources": [{"path": os.path.basename(src)}]}, f)

print("kaggle CLI:", shutil.which("kaggle"), "creds ~/.kaggle/kaggle.json:",
      os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")))

for attempt in (1, 2):
    cmd = ["kaggle", "datasets", "version", "-p", stage, "-m", "probe"] if attempt == 1 \
        else ["kaggle", "datasets", "create", "-p", stage]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"[{attempt}] {' '.join(cmd[:3])} rc={r.returncode}")
    print((r.stdout + r.stderr)[-800:])
    if r.returncode == 0:
        break

# обязательный контроль: датасет реально виден и файл в нём непустой
r = subprocess.run(["kaggle", "datasets", "files", f"{owner}/{slug}"], capture_output=True, text=True)
print("files rc=", r.returncode)
print((r.stdout + r.stderr)[-500:])
print("PROBE_DONE")
